<!-- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand -->
<a href="https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials"><img src="https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/brand/synapsa-commons-badge.png" alt="Synapsa Commons" height="36"></a>

Free, hands-on AI courses that run anywhere, from the team building [Synapsa](https://synapsa.realai.eu), an AI-native
learning platform.

© 2026 RealAI · free to learn from, share and adapt, not to sell ([CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)).
The notice at the end of this notebook says what you may and may not do.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T08-L01-agent-observability/lesson.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T08-L01-agent-observability/lesson.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master?labpath=lessons/T08-L01-agent-observability/lesson.ipynb)
[![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials)

This lesson needs Python 3.11 or newer.

In [ ]:
# --- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand ---
# Makes this notebook run anywhere. Every line is a no-op when the thing is already present,
# so a local clone pays nothing and an online notebook repairs itself.
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

COMMONS_PIP = []            # (import name, pinned pip spec) for what this lesson imports
COMMONS_SIBLINGS = []    # files that must sit beside the notebook
# A fork, a classroom mirror or an offline copy can serve the files from elsewhere by setting
# COMMONS_RAW_OVERRIDE before running this cell.
COMMONS_RAW = os.environ.get("COMMONS_RAW_OVERRIDE") or "https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/lessons/T08-L01-agent-observability/"

# Resolve siblings against the LESSON's own directory, not the working directory. A notebook
# has no __file__ and runs with cwd alongside itself; a grader imports this file from the repo
# root. Checking cwd blindly makes the grader think every sibling is missing and reach for the
# network -- which would put a download on a graded path.
try:
    COMMONS_DIR = Path(__file__).resolve().parent
except NameError:
    COMMONS_DIR = Path.cwd()


def commons_host() -> str:
    """Name the notebook service we are on. Used for the message, and for honest errors."""
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "Google Colab"
    except (ImportError, ValueError):
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "Kaggle"
    if os.environ.get("BINDER_SERVICE_HOST"):
        return "Binder"
    if os.environ.get("CODESPACES"):
        return "GitHub Codespaces"
    return "a local Python environment"


_missing = [pip for imp, pip in COMMONS_PIP if importlib.util.find_spec(imp) is None]
if _missing:
    print("installing " + ", ".join(_missing) + " ...")
    # pip everywhere a student is likely to be; uv-managed local venvs ship without pip.
    if importlib.util.find_spec("pip") is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    else:
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_missing],
                       check=True)
    importlib.invalidate_caches()

_fetched = []
for _name in COMMONS_SIBLINGS:
    if not (COMMONS_DIR / _name).exists():
        (COMMONS_DIR / _name).parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(COMMONS_RAW + _name, COMMONS_DIR / _name)
            _fetched.append(_name)
        except Exception as _e:  # Kaggle disables the internet by default; say so plainly
            raise RuntimeError(
                f"this lesson needs {_name} beside the notebook and could not fetch it "
                f"({_e}). On Kaggle, switch Internet on in the notebook settings panel "
                f"(Kaggle allows that only for phone-verified accounts); otherwise download it "
                f"from {COMMONS_RAW + _name} and upload it beside the notebook."
            ) from None

print("ready on " + commons_host() + ("; fetched " + ", ".join(_fetched) if _fetched else ""))
# --- END COMMONS LAUNCHER ---

# T08-L01 · Observability for agents: OpenTelemetry GenAI spans, from the agent loop to a cost-per-task report

**You will build:** the instrumentation that turns the T07 agent loop into OpenTelemetry
spans following the GenAI semantic conventions, an OTLP/JSON encoder and decoder written by
hand, and a report that recomputes steps, errors and cost per success from the exported spans
alone, then checks that report against T07-L03's own trace metrics.

**Time:** ~50 minutes · **Runs on:** a laptop CPU, no GPU, no download, no network ·
**Prerequisites:** T07-L01-agent-loop-from-scratch, T07-L03-evaluating-agents.

There is no language model and no OpenTelemetry package here. The agent is T07-L01's seeded
`ScriptedPolicy`, and the spans are built with the standard library in the OpenTelemetry data
model, so that everything you print is reproducible on any machine.

By the end you will be able to:

1. Implement the GenAI-convention attributes of a tool-execution span, including the
   `error.type` classification and the rule that a step which succeeded leaves its span
   status unset.
2. Instrument the T07-L01 agent loop so each episode is an `invoke_agent` span with one
   `execute_tool` child span per step, without changing a single random draw the agent makes.
3. Implement OTLP/JSON attribute encoding and decoding by hand, including the rule that
   64-bit integers travel as strings, and export one payload to two local sinks.
4. Recompute step counts, tool errors, recoveries and cost per success from the exported
   spans alone, in any arrival order, and verify they equal T07-L03's trace metrics exactly.
5. Measure the instrumentation overhead as spans, attributes and bytes per agent step, and
   measure how far cost per success drifts when only successful traces are kept.

<!-- COMMONS BEFORE v1 · generated from meta.yaml by tools/notebooks.py · do not edit by hand -->
## Before you start

**Lessons to do first:** `T07-L01`, `T07-L03`

**This lesson assumes you can already:**

- wrap a function call in try/except and a with block, and record or re-raise the exception
- read nested JSON (lists of dicts) and group records by a key to total a field
- turn two nanosecond timestamps since the Unix epoch into a duration

**New to some of that? These are good places to learn it** (free to read online first, then paid books). You do not need any of them if the list above already reads as familiar.

- [Traces](https://opentelemetry.io/docs/concepts/signals/traces/) · OpenTelemetry project · *free* — spans, parent and child, attributes and span status (Unset, Ok, Error): the data model section 3 and exercise 1 are written in
- [OpenTelemetry GenAI semantic conventions: agent spans (semantic-conventions-genai repository)](https://github.com/open-telemetry/semantic-conventions-genai/blob/main/docs/gen-ai/gen-ai-agent-spans.md) · OpenTelemetry project · *free* — the invoke_agent and execute_tool span definitions, their gen_ai attributes and error.type, which exercises 1 and 2 follow; the old opentelemetry.io page now points here
- [OTLP Specification: JSON Protobuf Encoding](https://opentelemetry.io/docs/specs/otlp/) · OpenTelemetry project · *free* — the JSON encoding section, including 64-bit integers carried as decimal strings, which exercises 3 and 4 implement by hand
- [OpenTelemetry for Python](https://opentelemetry.io/docs/languages/python/) · OpenTelemetry project · *free* — getting started with the real SDK and exporters, for the optional section 14 and for taking this to a real agent

In [ ]:
# Setup: everything the lesson needs, in one cell.
import contextlib
import io
import json
import os
import random
import signal
import sqlite3
import sys
import threading
import time
import traceback
from dataclasses import dataclass, field
from typing import Any, Callable, NamedTuple, Sequence

print("python", sys.version.split()[0])

SEED = 20260929          # this lesson's one fixed seed: every task, fault, retry delay, span id
                          # and timestamp derives from it, so re-running prints the same numbers.

_FAILED_CHECKS: list[str] = []
_STATUS: dict[str, str] = {}   # label -> "passed" | "failed" | "not started", latest run

# The exercises, in the order you meet them, and the functions each one asks you to write.
# The progress board at the foot of the notebook is built from this, and a cell that is
# waiting on an unfinished exercise names it from here.
_EXERCISES: dict[str, tuple[str, ...]] = {
    "exercise 1": ("tool_span_attributes",),
    "exercise 2": ("traced_call",),
    "exercise 3": ("encode_attributes",),
    "exercise 4": ("decode_attributes",),
    "exercise 5": ("report_from_spans",),
}


def _named(labels: list[str]) -> str:
    """["exercise 3"] -> "exercise 3 (encode_attributes)"; several -> "exercises 3, 6 and 8"."""
    if len(labels) == 1:
        return f"{labels[0]} ({', '.join(_EXERCISES[labels[0]])})"
    nums = [label.split()[-1] for label in labels]
    return "exercises " + ", ".join(nums[:-1]) + " and " + nums[-1]


class CheckTimedOut(BaseException):
    """A check ran past _CHECK_SECONDS. A BaseException, so an `except Exception` inside
    the code being checked cannot swallow it and keep looping."""


_CHECK_SECONDS = 2     # every check and demo here finishes in well under a second
_DEMO_SECONDS = 10


def _resident() -> float:
    """This process's resident memory now, in MiB (Linux: /proc; macOS: libproc); its peak so far elsewhere."""
    try:
        if sys.platform.startswith("linux"):
            with open("/proc/self/statm") as f:
                return int(f.read().split()[1]) * os.sysconf("SC_PAGE_SIZE") / 2**20
        if sys.platform == "darwin":
            import ctypes, struct
            info = ctypes.create_string_buffer(96)           # struct proc_taskinfo
            if ctypes.CDLL("/usr/lib/libproc.dylib").proc_pidinfo(os.getpid(), 4, 0, info, 96) == 96:
                return struct.unpack_from("QQ", info.raw)[1] / 2**20     # pti_resident_size
    except Exception:
        pass
    import resource
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / (2**20 if sys.platform == "darwin" else 2**10)


@contextlib.contextmanager
def _time_limit(seconds: float, what: str, memory_mib: float = 2048):
    """Turn code that never returns into a failed check that says so, instead of a notebook
    that hangs. It uses the operating system's alarm where there is one (Linux and macOS, in
    the main thread) and does nothing elsewhere. An alarm that was already running (the
    grader's own time limit) keeps its deadline if it is due first and is put back, with the
    time it had left, afterwards. A correct answer finishes far inside the limit, and nothing
    that is printed depends on it."""
    if not (all(hasattr(signal, name) for name in ("setitimer", "SIGALRM", "pthread_sigmask", "sigpending", "sigwait"))
            and threading.current_thread() is threading.main_thread()):
        yield
        return
    if getattr(_time_limit, "stopped", False):           # a stopped block's frames keep what they built in a
        _time_limit.stopped = False                      # reference cycle until now: free it, so it is not counted
        import gc
        gc.collect()                                     # as the growth of this block
    message = (f"{what} was still running after {seconds:g} s and was stopped: a "
               "loop that never ends? (a walk up parent ids that never reaches a "
               "root, a queue that never empties, a retry with no cap)")
    started = time.monotonic()
    used_mib = None
    if memory_mib:
        used_mib = _resident
        base_mib = used_mib()                            # resident memory now; only growth beyond it counts
    previous = signal.getsignal(signal.SIGALRM)
    outer_left, outer_every = signal.getitimer(signal.ITIMER_REAL)   # an alarm already running
    # One alarm at a time, never a repeating one. next: when ours is due; outer: when the alarm we
    # found is due (every: its interval); unseen: an alarm is armed and has not reached _fire yet.
    state = {"next": started + seconds, "outer": started + outer_left if outer_left > 0 else None,
             "every": outer_every, "unseen": False, "fired": False, "done": False, "message": message, "count": 0, "budget": 0,
             "traced": False, "tracer": None, "tool": None, "at": None, "thread": threading.get_ident(),
             "me": sys._getframe(), "owner": None}
    owner = state["me"].f_back
    while owner is not None and owner.f_globals is vars(contextlib):    # contextlib's __enter__
        owner = owner.f_back
    state["owner"], state["at"] = owner, owner.f_lineno   # the frame running the `with`, and its line

    def _arm(now, soonest=0.001):                        # one shot, at whichever deadline is next
        due = [t for t in (state["next"], state["outer"]) if t is not None]
        if used_mib and state["next"] is not None and not state["fired"]:
            due.append(now + 0.1)                        # and look at the memory every 0.1 s
        if due:
            signal.setitimer(signal.ITIMER_REAL, max(min(due) - now, soonest))
            state["unseen"] = True

    def _in_guard(frame):                                # is the alarm landing in a guard's own set-up, exit
        while frame is not None and frame is not state["owner"]:     # or clean-up (this one's or another's:
            code = frame.f_code                          # guards nest), or in contextlib's __exit__ under
            if (frame is state["me"] or code in _own_code or code.co_name in _GUARDS   # a `with`?
                    or frame.f_globals.get("__name__") == "contextlib"):
                return True
            frame = frame.f_back
        return False

    def _fire(signum, frame):
        state["unseen"] = False
        if state["done"]:
            return                                       # a late alarm of ours: already handled
        if _in_guard(frame):                             # never raise inside the guard: it is
            _arm(time.monotonic(), 0.01)                 # finishing; look again shortly
            return
        now = time.monotonic()
        if state["outer"] is not None and now >= state["outer"] - 0.005:
            every = state["every"]                       # the outer alarm is due: act as it would
            state["outer"] = state["outer"] + every * (1 + (now - state["outer"]) // every) if every else None
            try:
                if callable(previous):
                    previous(signum, frame)
                elif previous != signal.SIG_IGN:         # its default action ends the process
                    signal.signal(signal.SIGALRM, signal.SIG_DFL)
                    os.kill(os.getpid(), signal.SIGALRM)
            except BaseException:
                state["next"] = None                     # the outer one is stopping the block
                raise
            finally:
                left, every = signal.getitimer(signal.ITIMER_REAL)
                if left > 0:                             # the outer handler armed its next alarm
                    state["outer"], state["every"] = time.monotonic() + left, every
                _arm(time.monotonic())
            now = time.monotonic()
        grown = used_mib() - base_mib if used_mib and not state["fired"] else 0
        if grown > memory_mib and state["next"] is not None:       # out of memory before out of time
            state["message"] = (f"{what} was using over {memory_mib:g} MiB more memory than when it started, "
                                f"after {now - started:.1f} s: a loop that keeps adding to a list or a string "
                                "never stops; give every loop a cap")
            state["next"] = now
        if state["next"] is not None and now >= state["next"] - 0.005:
            swallowed = state["fired"]                   # a second stop: the first did not end the block
            state["fired"] = True
            state["count"] += 1
            state["next"] = now + 0.25                   # again, in case a bare `except:` swallows it
            _arm(now)
            if swallowed:
                state["budget"] = state["count"] - 1     # the tracer raises this many times, then leaves clean-up alone
                _trace_from(frame)
            raise CheckTimedOut(state["message"], state)
        _arm(now)

    def _trace(frame, event, arg):                       # raises the stop at the next line of code that
        if state["done"] or _in_guard(frame):            # runs: inside the handler that swallowed the last
            return None                                  # (the interpreter then unsets it; the next alarm
        if event == "line":                              # sets it again)
            raise CheckTimedOut(state["message"], state)
        return _trace

    def _line(code, line):                               # Python 3.12+: sys.monitoring, which is not unset
        if state["done"] or state["budget"] <= 0 or threading.get_ident() != state["thread"]:   # not unset when a
            return                                       # callback raises: the next line the code runs raises too, handlers
        frame = sys._getframe(1)                         # included (this thread only), but only `budget` times:
        if _in_guard(frame) or (frame is state["owner"] and line == state["at"]):   # clean-up that runs once the stop has
            return                                       # left the swallowing loops must not be interrupted again
        state["budget"] -= 1                             # (not the `with` line: its exit runs the guard)
        raise CheckTimedOut(state["message"], state)

    def _trace_from(frame):                              # trace the running frames, and the ones called next
        if not state["traced"]:
            state["traced"] = True
            mon = getattr(sys, "monitoring", None)
            for tool in (5, 4, 3, 2, 1, 0) if mon is not None else ():
                try:
                    mon.use_tool_id(tool, "commons-time-limit")
                except ValueError:                       # a debugger or a profiler has this one
                    continue
                state["tool"] = tool
                mon.register_callback(tool, mon.events.LINE, _line)
                mon.set_events(tool, mon.events.LINE)
                return
            state["tracer"] = sys.gettrace()
            sys.settrace(_trace)
        while frame is not None and frame is not state["owner"]:
            frame.f_trace = _trace
            frame = frame.f_back

    _GUARDS = ("_time_limit", "_deadline", "_resident", "_time_limit", "_fire", "_arm", "_trace", "_trace_from", "_line", "_in_guard")
    _own_code = (_fire.__code__, _arm.__code__, _trace.__code__, _trace_from.__code__, _line.__code__)

    signal.signal(signal.SIGALRM, _fire)
    _arm(time.monotonic())
    try:
        try:
            yield
        finally:
            state["done"] = True                         # from here on an alarm of ours only counts
            if state["traced"]:
                state["traced"] = False
                if state["tool"] is not None:
                    sys.monitoring.set_events(state["tool"], 0)
                    sys.monitoring.register_callback(state["tool"], sys.monitoring.events.LINE, None)
                    sys.monitoring.free_tool_id(state["tool"])
                else:
                    sys.settrace(state["tracer"])        # the tracer that was there before, if any
            mask = signal.pthread_sigmask(signal.SIG_BLOCK, {signal.SIGALRM})
            try:
                if signal.setitimer(signal.ITIMER_REAL, 0)[0] == 0 and state["unseen"]:
                    # Ours went off and has not reached _fire: it is pending here, or on its way to
                    # another thread. Take it before the old handler (by default, one that ends the
                    # process) is back.
                    give_up = time.monotonic() + 0.5
                    while state["unseen"] and time.monotonic() < give_up:
                        if signal.SIGALRM in signal.sigpending():
                            signal.sigwait({signal.SIGALRM})
                            state["unseen"] = False
                        else:
                            time.sleep(0.001)            # another thread has it: _fire runs next
                signal.signal(signal.SIGALRM, signal.SIG_DFL if previous is None else previous)
                outer = state["outer"]                   # re-arm the alarm we found, last
                signal.setitimer(signal.ITIMER_REAL, *((max(outer - time.monotonic(), 0.001), state["every"])
                                                       if outer is not None else (0, 0)))
            finally:
                signal.pthread_sigmask(signal.SIG_SETMASK, mask)
                state["me"] = state["owner"] = None
    except CheckTimedOut as exc:
        if exc.args[-1] is not state:                    # an enclosing guard's: not ours to report
            raise
        _time_limit.stopped = True
        raise CheckTimedOut(state["message"]) from None
    if state["fired"]:                                   # the block caught our alarm and carried on
        _time_limit.stopped = True
        raise CheckTimedOut(state["message"])


def _try(label: str, check: Callable[[], None], needs: tuple[str, ...] = ()) -> None:
    """Run a check, or a demo that depends on your code, without derailing the notebook.

    A stub you have not filled in yet simply says so. A wrong answer prints the check's own
    message — which names the likely mistake — and the notebook carries on, so one broken
    exercise never hides the feedback on the others. A demo names the exercises it `needs`:
    until each has passed its check, the demo says which one it is waiting for and skips.
    Nothing is swallowed: every outcome is recorded in `_STATUS` for the progress board at the
    foot of the notebook, and every failure in `_FAILED_CHECKS`, which ends a script run
    non-zero.
    """
    waiting = [name for name in _EXERCISES   # in the order you meet them
               if name in needs and _STATUS.get(name) != "passed"]
    if waiting:
        _STATUS[label] = "not started"
        print(f"{label}: skipped — needs {_named(waiting)} to pass first.")
        return
    try:
        with _time_limit(_CHECK_SECONDS if label in _EXERCISES else _DEMO_SECONDS, label):
            check()
    except CheckTimedOut as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: FAILED — {exc}")
    except NotImplementedError as exc:
        _STATUS[label] = "not started"
        stub = traceback.extract_tb(exc.__traceback__)[-1].name   # the frame that raised
        owner = [name for name, funcs in _EXERCISES.items() if stub in funcs and name != label]
        if owner:
            print(f"{label}: skipped — needs {_named(owner)} first.")
        elif label in _EXERCISES:
            print(f"{label}: not implemented yet — fill in {stub}() above, then re-run "
                  "this cell.")
        else:
            print(f"{label}: skipped — {stub}() is not implemented yet.")
    except AssertionError as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: FAILED — {exc}")
    except Exception as exc:  # a half-finished implementation raising something else
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: raised {type(exc).__name__}: {exc}")
    else:
        _STATUS[label] = "passed"

## 1. Why traces, and which conventions this lesson follows

A multi-step agent is a non-deterministic program whose interesting failures happen in the
middle: a tool times out and is retried, a call is malformed, the loop repeats itself. A
success rate says none of that. A **trace** does: one tree per task, with a **span** for the
episode and a child span for every tool call, each carrying attributes and a status.

OpenTelemetry (OTel) defines the span data model and a wire format, OTLP. Its **GenAI
semantic conventions** say what to call things: the span for one agent run is
`invoke_agent {name}`, the span for one tool call is `execute_tool {tool}`, and attributes
such as `gen_ai.operation.name` and `error.type` have fixed spellings, so that any backend
that understands the conventions can draw your agent without being told about it.

Two facts here move fast, so they are pinned and dated rather than remembered. The GenAI
conventions have moved out of the main OTel semantic-conventions repository into their own,
and on the date printed below they were marked as being in development. That word is a
warning: attribute names, values and requirement levels can still change, and may already
have changed since that date. Everything this lesson calls "the convention" is read from the
pinned text, not from whatever the repository says today. The next cell prints the pin.

In [ ]:
# Maintainer's note: this repository has published no release, so a COMMIT is the version. To
# refresh the lesson, read the two files below at the repository's current commit (raw URL:
# https://raw.githubusercontent.com/<repository>/<commit>/<file>), compare every attribute this
# lesson uses (gen_ai.operation.name, gen_ai.agent.name, gen_ai.tool.name, gen_ai.tool.type,
# gen_ai.tool.call.arguments, error.type and the span names), then change the values here AND the
# source_url and quote of every GenAI entry in claims.yaml, which carry the commit hash as well.
SEMCONV_PIN = {
    "repository": "open-telemetry/semantic-conventions-genai",
    "commit": "e57c543b4889619eb2a05702471937db5119165d",
    "commit_date": "2026-09-24",
    "read_on": "2026-09-29",
    "status": "Development",
    "files_read": "docs/gen-ai/gen-ai-spans.md, docs/gen-ai/gen-ai-agent-spans.md",
    "core_conventions": "v1.44.0",
    "otlp_specification": "1.11.0",
}
print("GenAI semantic conventions, as of", SEMCONV_PIN["read_on"])
for _k, _v in SEMCONV_PIN.items():
    print(f"  {_k:<32} {_v}")
print(f"On {SEMCONV_PIN['read_on']} the conventions had no released version of their own, so "
      "this lesson pins the commit it read; the repository may have moved on since.")
print(f"On that date their status was {SEMCONV_PIN['status']}: names and requirement levels may "
      "have changed since, so check the repository's current text before you copy this into "
      "production.")
print("Where these facts come from is in claims.yaml, each with a source and an access date.")

## 2. The agent, carried forward

The agent is T07-L01's, copied here and not imported, because a lesson has to run alone:
a seeded lookup-chain task, two tools (`lookup`, `submit`), validation, capped retries with
jittered backoff, a step budget, a cost budget and a stuck-loop detector. The behaviour is
unchanged; the docstrings are shortened. Two small pieces come from T07-L03: the trajectory
metrics and the failure taxonomy that this lesson's span report must reproduce.

One number needs a definition. T07-L01 counts **cost** in tool-call attempts: every call the
harness makes, retries included. It is the only price in this lesson because there is no
model to bill for tokens. The GenAI conventions have attributes for token usage on model
calls, and a real deployment would multiply those by a price list; here an attempt is the
unit, and cost per success is total attempts over the number of successes.

In [ ]:
class TransientToolError(Exception):
    """Raised by a tool for a failure worth retrying (a timeout, a dropped connection)."""


class ToolResult(NamedTuple):
    ok: bool
    output: Any
    error: str | None
    transient: bool     # only meaningful when ok is False: worth a retry, or not?


class ToolRegistry(NamedTuple):
    tools: dict[str, Callable[..., Any]]
    schemas: dict[str, dict]


TOOL_SCHEMAS: dict[str, dict] = {
    "lookup": {"type": "object", "properties": {"key": {"type": "string"}},
              "required": ["key"]},
    "submit": {"type": "object", "properties": {"value": {"type": "integer"}},
              "required": ["value"]},
}


class ChainTask(NamedTuple):
    task_id: int
    start_key: str
    key_sequence: list[str]          # the correct keys, in order; key_sequence[0] == start_key
    chain: dict[str, Any]            # key -> next key (str), or the terminal value (int)
    target: int
    length: int                      # == len(key_sequence)


def generate_chain_task(length: int, seed: int) -> ChainTask:
    """One deterministic lookup chain of the given length."""
    rng = random.Random(seed)
    keys = [f"k{i}_{rng.randrange(1_000_000):06d}" for i in range(length)]
    chain: dict[str, Any] = {keys[i]: keys[i + 1] for i in range(length - 1)}
    target = rng.randint(100, 999)
    chain[keys[-1]] = target
    return ChainTask(task_id=seed, start_key=keys[0], key_sequence=keys, chain=chain,
                     target=target, length=length)


def generate_task_suite(lengths: Sequence[int], n_per_length: int, seed: int) -> list[ChainTask]:
    """`n_per_length` tasks at each length in `lengths`, all deterministic given `seed`."""
    tasks, task_id = [], 0
    for length in lengths:
        for _ in range(n_per_length):
            tasks.append(generate_chain_task(length, seed=seed * 1000 + task_id))
            task_id += 1
    return tasks


def make_registry(chain: dict, target: int, transient_rate: float,
                  rng: random.Random) -> ToolRegistry:
    """A registry over ONE task's chain; `lookup` fails transiently at `transient_rate`."""
    def lookup(key):
        if rng.random() < transient_rate:
            raise TransientToolError("lookup timed out")
        if key not in chain:
            raise ValueError(f"unknown key {key!r}")
        val = chain[key]
        return {"next_key": val} if isinstance(val, str) else {"value": val}

    def submit(value):
        return {"correct": value == target}

    return ToolRegistry(tools={"lookup": lookup, "submit": submit}, schemas=TOOL_SCHEMAS)


_ARG_TYPES = {"string": str, "integer": int, "number": (int, float), "boolean": bool,
             "array": list, "object": dict}


def validate_args(schema: dict, args: dict) -> list[str]:
    """Check `args` against a documented subset of JSON Schema, by hand."""
    errors: list[str] = []
    for name in schema.get("required", []):
        if name not in args:
            errors.append(f"missing required argument {name!r}")
    properties = schema.get("properties", {})
    for name, value in args.items():
        prop = properties.get(name)
        if prop is None:
            continue          # this documented subset does not forbid extra arguments
        expected = prop.get("type")
        py_type = _ARG_TYPES.get(expected)
        if py_type is not None:
            reject = (isinstance(value, bool) if expected in ("integer", "number")
                      else False)
            if reject or not isinstance(value, py_type):
                errors.append(f"argument {name!r} should be of type {expected!r}, got {value!r}")
                continue
        if "enum" in prop and value not in prop["enum"]:
            errors.append(f"argument {name!r} must be one of {prop['enum']!r}, got {value!r}")
    return errors


def call_tool(registry: ToolRegistry, name: str, args: dict, validate: bool = True) -> ToolResult:
    """Execute one tool call through `registry`, never letting an exception escape."""
    if name not in registry.tools:
        return ToolResult(ok=False, output=None, error=f"unknown tool {name!r}", transient=False)
    schema = registry.schemas.get(name, {})
    if validate:
        errors = validate_args(schema, args)
        if errors:
            return ToolResult(ok=False, output=None,
                              error="invalid arguments: " + "; ".join(errors), transient=False)
        try:
            return ToolResult(ok=True, output=registry.tools[name](**args), error=None,
                              transient=False)
        except TransientToolError as exc:
            return ToolResult(ok=False, output=None, error=str(exc), transient=True)
        except Exception as exc:
            return ToolResult(ok=False, output=None,
                              error=f"tool {name!r} raised {type(exc).__name__}: {exc}",
                              transient=False)
    try:
        return ToolResult(ok=True, output=registry.tools[name](**args), error=None,
                          transient=False)
    except Exception as exc:
        return ToolResult(ok=False, output=None,
                          error=f"tool {name!r} raised {type(exc).__name__}: {exc}",
                          transient=True)


BASE_DELAY = 0.05    # seconds (simulated — nothing in this lesson actually sleeps)
CAP_DELAY = 2.0


def backoff_delay(attempt: int, base: float, cap: float, rng: random.Random) -> float:
    """Full-jitter capped exponential backoff. `attempt` is 0 for the FIRST retry."""
    ceiling = min(cap, base * (2 ** attempt))
    return rng.uniform(0.0, ceiling)


def call_tool_with_retries(registry: ToolRegistry, name: str, args: dict, max_retries: int,
                           rng: random.Random, validate: bool = True) -> tuple[ToolResult, int, float]:
    """Call a tool, retrying only a TRANSIENT failure, up to `max_retries` times.

    Returns (result, attempts, total_backoff): attempts is the number of calls made.
    """
    attempt = 0
    total_backoff = 0.0
    result = call_tool(registry, name, args, validate=validate)
    while not result.ok and result.transient and attempt < max_retries:
        total_backoff += backoff_delay(attempt, BASE_DELAY, CAP_DELAY, rng)
        attempt += 1
        result = call_tool(registry, name, args, validate=validate)
    return result, attempt + 1, total_backoff


def is_stuck(trace: list[dict], window: int) -> bool:
    """True if the LAST `window` steps of `trace` are all the same action with the same outcome."""
    if len(trace) < window:
        return False
    last = trace[-window:]

    def signature(entry):
        return entry["tool"], tuple(sorted(entry["args"].items())), entry["ok"]

    first = signature(last[0])
    return all(signature(entry) == first for entry in last)


def budget_status(step: int, cost_spent: int, max_steps: int, max_cost: int) -> str | None:
    """Which budget, if any, is exhausted — checked in this fixed order: step, then cost."""
    if step >= max_steps:
        return "step_budget"
    if cost_spent >= max_cost:
        return "cost_budget"
    return None


FAULT_KINDS = ("malformed_args", "unknown_tool", "hallucinated_submit", "repeat_last")


class AgentTrace(NamedTuple):
    steps: list[dict]
    status: str          # "success" | "wrong_answer" | "step_budget" | "cost_budget" | "stuck"
    cost_spent: int
    n_steps: int


class ScriptedPolicy:
    """A deterministic, seeded stand-in for "the model" that proposes actions."""

    def __init__(self, plan: list[tuple[str, dict]], final_action: tuple[str, dict],
                fault_rate: float, rng: random.Random):
        self.plan = list(plan)
        self.final_action = final_action
        self.fault_rate = fault_rate
        self.rng = rng
        self.cursor = 0

    def _expected(self) -> tuple[str, dict]:
        return self.plan[self.cursor] if self.cursor < len(self.plan) else self.final_action

    def next_action(self, trace: list[dict]) -> tuple[str, dict]:
        if trace:
            last = trace[-1]
            if (self.cursor < len(self.plan) and last["ok"] and last["tool"] == "lookup"
                    and last["args"] == self.plan[self.cursor][1]):
                self.cursor += 1
        correct_tool, correct_args = self._expected()
        if self.rng.random() >= self.fault_rate:
            return correct_tool, dict(correct_args)
        fault = self.rng.choice(FAULT_KINDS)
        target = self.final_action[1]["value"]
        if fault == "malformed_args":
            return ("lookup", {}) if correct_tool == "lookup" else ("submit", {"value": str(target)})
        if fault == "unknown_tool":
            return "look_up", dict(correct_args)
        if fault == "hallucinated_submit":
            guess = target
            while guess == target:
                guess = self.rng.randint(0, 999)
            return "submit", {"value": guess}
        # "repeat_last"
        if trace:
            return trace[-1]["tool"], dict(trace[-1]["args"])
        return "lookup", {}     # nothing to repeat yet on the very first decision


class TrajectoryMetrics(NamedTuple):
    n_episodes: int
    mean_steps: float
    tool_errors: int
    recoveries: int
    cost_per_success: float


def trajectory_metrics(traces: list[AgentTrace]) -> TrajectoryMetrics:
    """T07-L03's trajectory metrics, read straight from the agent's own traces."""
    n_episodes = len(traces)
    mean_steps = sum(tr.n_steps for tr in traces) / n_episodes
    tool_errors = sum(1 for tr in traces for step in tr.steps if not step["ok"])
    recoveries = sum(1 for tr in traces for idx, step in enumerate(tr.steps)
                     if not step["ok"] and idx < len(tr.steps) - 1)
    n_success = sum(1 for tr in traces if tr.status == "success")
    total_cost = sum(tr.cost_spent for tr in traces)
    cost_per_success = total_cost / n_success if n_success else float("inf")
    return TrajectoryMetrics(n_episodes, mean_steps, tool_errors, recoveries, cost_per_success)


def failure_taxonomy_from_traces(traces: list[AgentTrace]) -> dict[str, int]:
    """Counts of each non-success status, highest count first, ties in first-seen order."""
    counts: dict[str, int] = {}
    for tr in traces:
        if tr.status != "success":
            counts[tr.status] = counts.get(tr.status, 0) + 1
    return dict(sorted(counts.items(), key=lambda kv: -kv[1]))


LENGTHS = [1, 2, 3, 4, 6, 8, 10, 14, 18, 24]
N_PER_LENGTH = 20
FAULT_RATE = 0.05
TRANSIENT_RATE = 0.12
MAX_RETRIES = 3
STUCK_WINDOW = 3
MAX_STEPS = 80
MAX_COST = 200

_SUITE = generate_task_suite(LENGTHS, N_PER_LENGTH, seed=SEED)
print(f"suite: {len(_SUITE)} tasks over lengths {LENGTHS}, {N_PER_LENGTH} tasks per length, "
      f"seed {SEED}")

## 3. Spans, a tracer and exporters, in the OpenTelemetry data model

The OpenTelemetry SDK is a large package, and what a span *is* fits on a page. A span has
a trace id (16 bytes, shown as 32 hex characters), a span id (8 bytes, 16 hex characters),
an optional parent span id, a name, a kind, start and end times in nanoseconds, a bag of
attributes and a status. Spans of one trace share the trace id; the parent id makes the tree.

`Tracer` below is a small, given implementation of the same idea: `with tracer.span(...)`
starts a span as a child of whichever span is current, ends it when the block exits (even on
an exception, which it records), and when the *root* span ends it hands every span of that
trace to an **exporter**, in the order they ended, children before their parent, because the
SDK's span processors are told about a span only after it has ended.

Two things are deliberately not the real thing. Ids come from a seeded generator of their own,
and the clock is a counter that ticks a millisecond per event, so this notebook prints the
same spans on every machine and the tracer never touches the random streams the agent uses.

In [ ]:
SPAN_KIND_INTERNAL = 1                       # SPAN_KIND_INTERNAL in the OTLP schema
STATUS_UNSET, STATUS_OK, STATUS_ERROR = 0, 1, 2   # StatusCode in the OTLP schema
_CLOCK_START_NS = 1_790_000_000_000_000_000
_TICK_NS = 1_000_000


@dataclass
class Span:
    name: str
    kind: int
    trace_id: str                      # 32 lowercase hex characters
    span_id: str                       # 16 lowercase hex characters
    parent_span_id: str | None         # None for a root span
    start_ns: int
    end_ns: int | None = None
    attributes: dict[str, Any] = field(default_factory=dict)
    status_code: int = STATUS_UNSET
    status_message: str = ""

    def set_attribute(self, key: str, value: Any) -> None:
        self.attributes[key] = value

    def set_attributes(self, attributes: dict[str, Any]) -> None:
        self.attributes.update(attributes)

    def set_status(self, code: int, message: str = "") -> None:
        """A description is only allowed with the Error code, as in the OTel trace API."""
        self.status_code = code
        self.status_message = message if code == STATUS_ERROR else ""


class Tracer:
    """Starts spans, keeps the current one, and exports a finished trace to `exporter`."""

    def __init__(self, seed: str, exporter: Any):
        self._ids = random.Random(f"{seed}:otel-ids")   # its own stream: never the agent's
        self._now = _CLOCK_START_NS
        self._stack: list[Span] = []
        self._finished: list[Span] = []
        self.exporter = exporter

    def _tick(self) -> int:
        self._now += _TICK_NS
        return self._now

    def current_span(self) -> Span | None:
        return self._stack[-1] if self._stack else None

    @contextlib.contextmanager
    def span(self, name: str, kind: int = SPAN_KIND_INTERNAL,
             attributes: dict[str, Any] | None = None):
        parent = self.current_span()
        trace_id = parent.trace_id if parent else format(self._ids.getrandbits(128), "032x")
        sp = Span(name=name, kind=kind, trace_id=trace_id,
                  span_id=format(self._ids.getrandbits(64), "016x"),
                  parent_span_id=parent.span_id if parent else None,
                  start_ns=self._tick(), attributes=dict(attributes or {}))
        self._stack.append(sp)
        try:
            yield sp
        except BaseException as exc:      # record the exception on the span, then re-raise it
            sp.set_status(STATUS_ERROR, str(exc))
            sp.attributes.setdefault("error.type", type(exc).__name__)
            raise
        finally:
            sp.end_ns = self._tick()
            self._stack.pop()
            self._finished.append(sp)
            if parent is None:
                batch, self._finished = self._finished, []
                self.exporter.export(batch)


class ListExporter:
    """Keeps finished spans as Python objects. Used by the checks and the printed tree."""

    def __init__(self) -> None:
        self.spans: list[Span] = []

    def export(self, spans: list[Span]) -> None:
        self.spans.extend(spans)


class FanOutExporter:
    """One instrumentation, several exporters: every batch goes to each of them."""

    def __init__(self, exporters: list[Any]):
        self.exporters = exporters

    def export(self, spans: list[Span]) -> None:
        for exporter in self.exporters:
            exporter.export(spans)


AGENT_NAME = "chain-walker"
SERVICE_NAME = "commons-t08-l01"

## 4. Exercise 1 — `tool_span_attributes()`

Every tool call becomes one `execute_tool {tool}` span. This exercise decides what goes on
it. Follow the conventions where they have a name for the thing, and add your own attribute
only where they have none. The conventions say nothing about a retry count or a step index,
so those two carry the prefix `commons.` (OTel recommends prefixing an application-specific
name with your application's name, and not borrowing one of its own namespaces as the prefix).

The convention also asks for `error.type` when the operation ended in an error, as a
low-cardinality identifier, not as free text: an error message with a key in it would give a
dashboard a new series for every task. Classify the failure from the `ToolResult`.

<details><summary>💡 Hint 1 — what to think about</summary>

Start from what is always true of a tool span: which operation it is, which tool, what kind
of tool, which step of the episode, how many attempts it took, how long the retries would
have waited. Then ask which of those values only exist when the step failed. A step whose
first attempts timed out and whose last attempt worked did not end in an error.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Build the always-present attributes first, then add `error.type` only when the result is not
ok. To pick its value, look at whether the result is flagged transient before you look at the
wording of the message, and match the start of the message for the two harness-made errors
rather than searching for a word anywhere inside it. Anything else takes the convention's own
fallback value for "no better name".
</details>

In [ ]:
def tool_span_attributes(step_index: int, tool: str, result: ToolResult, attempts: int,
                         backoff: float) -> dict[str, Any]:
    """The attributes of one `execute_tool` span.

    Always present: "gen_ai.operation.name" = "execute_tool", "gen_ai.tool.name" = `tool`,
    "gen_ai.tool.type" = "function", "commons.step.index" = `step_index`,
    "commons.tool.attempts" = `attempts`, "commons.tool.backoff_seconds" = `backoff`.

    Only when `result.ok` is False, add "error.type": "timeout" if `result.transient`, else
    "unknown_tool" if the error message starts with "unknown tool", else "invalid_arguments"
    if it starts with "invalid arguments", else "_OTHER". A step that needed retries but ended
    ok has no "error.type" at all.

    Returns a dict; it never contains None.

    Example:
        >>> ok = ToolResult(ok=True, output={"value": 7}, error=None, transient=False)
        >>> a = tool_span_attributes(2, "lookup", ok, attempts=1, backoff=0.0)
        >>> a["gen_ai.operation.name"], a["gen_ai.tool.name"], "error.type" in a
        ('execute_tool', 'lookup', False)
        >>> bad = ToolResult(ok=False, output=None, error="lookup timed out", transient=True)
        >>> tool_span_attributes(2, "lookup", bad, attempts=4, backoff=0.3)["error.type"]
        'timeout'
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_tool_span_attributes() -> None:
    ok = ToolResult(ok=True, output={"value": 7}, error=None, transient=False)
    a = tool_span_attributes(2, "lookup", ok, 1, 0.0)
    assert a.get("gen_ai.operation.name") == "execute_tool", (
        f"gen_ai.operation.name is {a.get('gen_ai.operation.name')!r}; the convention's value "
        "for a tool call is execute_tool"
    )
    assert a.get("gen_ai.tool.name") == "lookup" and a.get("gen_ai.tool.type") == "function", (
        "gen_ai.tool.name must be the tool's own name and gen_ai.tool.type must be 'function'"
    )
    assert a.get("commons.step.index") == 2 and a.get("commons.tool.attempts") == 1, (
        "commons.step.index and commons.tool.attempts must carry the step index and the "
        "attempts you were given, as ints"
    )
    assert "error.type" not in a, (
        "a step that succeeded must have no error.type key at all — not None, not an empty "
        "string: a backend would count the key's presence as an error"
    )
    for tool, error, transient, want in [
            ("lookup", "lookup timed out", True, "timeout"),
            ("look_up", "unknown tool 'look_up'", False, "unknown_tool"),
            ("lookup", "invalid arguments: missing required argument 'key'", False,
             "invalid_arguments")]:
        bad = ToolResult(ok=False, output=None, error=error, transient=transient)
        got = tool_span_attributes(0, tool, bad, 1, 0.0).get("error.type")
        assert got == want, (
            f"a failed step with error {error!r} (transient={transient}) got error.type "
            f"{got!r}, expected {want!r}"
        )
    near = ToolResult(ok=False, output=None, error="unknown key 'k1'", transient=False)
    got = tool_span_attributes(0, "lookup", near, 1, 0.0).get("error.type")
    assert got == "_OTHER", (
        f"got {got!r} for 'unknown key ...': only a message that starts with the whole phrase "
        "'unknown tool' is the harness's unknown-tool error"
    )
    other = ToolResult(ok=False, output=None, error="tool 'lookup' raised ValueError: "
                       "unknown key 'k1'", transient=False)
    got = tool_span_attributes(0, "lookup", other, 1, 0.0).get("error.type")
    assert got == "_OTHER", (
        f"an error that is neither transient nor one of the two harness errors must take the "
        f"fallback value, got {got!r} — match the START of the message, not a word inside it"
    )
    for n_attempts in (1, 2, 3, 4, 5):
        retried = tool_span_attributes(1, "lookup", ok, n_attempts, 0.25)
        assert "error.type" not in retried and retried["commons.tool.attempts"] == n_attempts, (
            f"a step that needed {n_attempts} attempts but ended ok has no error.type; the "
            "retries show only in commons.tool.attempts"
        )
    assert retried.get("commons.tool.backoff_seconds") == 0.25, (
        "commons.tool.backoff_seconds must carry the total backoff you were given"
    )
    midway = ToolResult(ok=False, output=None, error="tool 'x' raised RuntimeError: invalid "
                        "arguments upstream", transient=False)
    got = tool_span_attributes(0, "lookup", midway, 1, 0.0).get("error.type")
    assert got == "_OTHER", (
        f"got {got!r}: the words 'invalid arguments' in the MIDDLE of a message do not make it "
        "the harness's own validation error; match the start of the message"
    )
    both = ToolResult(ok=False, output=None, error="unknown tool 'x'", transient=True)
    got = tool_span_attributes(0, "x", both, 1, 0.0).get("error.type")
    assert got == "timeout", (
        f"got {got!r} for a result flagged transient: the transient flag decides before the "
        "wording of the message does"
    )


_try("exercise 1", _check_tool_span_attributes)

## 5. Exercise 2 — `traced_call()`

This is the instrumentation itself: one tool call, wrapped in one span. The wrapper sits
between the loop and `call_tool_with_retries`, and its contract is what makes traces
trustworthy. The tool must run *inside* the span, so the span's duration and any spans the
tool starts belong to it. There is **one span per step**, however many attempts the step took;
the attempts are an attribute. And a span that ended without an error leaves its status
unset: the tracing API reserves an explicit OK for a human who has validated the outcome,
and the conventions say that errors which were retried and recovered should not be recorded
on the operation that ended well.

<details><summary>💡 Hint 1 — what to think about</summary>

Which object gives you a child span of "whatever span is current"? What has to happen while
the block is open, and what can only be known after the call returns? Your function must
return exactly what `call_tool_with_retries` returns, and must use the `rng` you were given
for the retries and nothing else, or an instrumented run stops matching an uninstrumented one.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Open a span named after the convention (the operation and the tool, separated by a space),
make the call inside it, then set the attributes from exercise 1 on the span. Set the status
to Error, with the tool's error text as its description, only when the result is not ok.
Leave the status alone otherwise. Return the call's own triple.
</details>

In [ ]:
def traced_call(tracer: Tracer, registry: ToolRegistry, step_index: int, tool: str, args: dict,
                max_retries: int, rng: random.Random,
                validate: bool = True) -> tuple[ToolResult, int, float]:
    """Run one tool call inside one `execute_tool {tool}` span; return the call's own triple.

    The span is named f"execute_tool {tool}", has kind SPAN_KIND_INTERNAL, and is a child of
    the tracer's current span. The call to `call_tool_with_retries(registry, tool, args,
    max_retries, rng, validate=validate)` happens INSIDE the span; its attributes come from
    `tool_span_attributes(step_index, tool, result, attempts, backoff)`. If the result is not
    ok the span status is STATUS_ERROR with `result.error` as the message; otherwise the status
    is left unset (not STATUS_OK). One span per call, however many attempts it took.

    Returns `(result, attempts, total_backoff)` exactly as `call_tool_with_retries` does.

    Example:
        >>> tracer = Tracer("doc", ListExporter())
        >>> registry = make_registry({"a": 1}, 1, transient_rate=0.0, rng=random.Random(0))
        >>> with tracer.span("invoke_agent demo"):
        ...     result, attempts, backoff = traced_call(
        ...         tracer, registry, 0, "lookup", {"key": "a"}, 2, random.Random(0))
        >>> [(s.name, s.status_code) for s in tracer.exporter.spans]
        [('execute_tool lookup', 0), ('invoke_agent demo', 0)]
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_traced_call() -> None:
    exporter = ListExporter()
    tracer = Tracer("check", exporter)
    registry = make_registry({"a": "b", "b": 5}, 5, transient_rate=0.0, rng=random.Random(0))
    with tracer.span("invoke_agent check") as root:
        result, attempts, backoff = traced_call(tracer, registry, 4, "lookup", {"key": "a"}, 2,
                                                random.Random(0))
    assert result.ok and attempts == 1, (
        "traced_call must return the same (result, attempts, backoff) triple that "
        "call_tool_with_retries returns"
    )
    tool_spans = [s for s in exporter.spans if s.name.startswith("execute_tool")]
    assert len(tool_spans) == 1 and tool_spans[0].name == "execute_tool lookup", (
        f"expected one span named 'execute_tool lookup', got {[s.name for s in exporter.spans]}"
        " — the name is the operation, a space, then the tool"
    )
    span = tool_spans[0]
    assert span.kind == SPAN_KIND_INTERNAL, f"kind is {span.kind}; a tool call is INTERNAL (1)"
    assert span.parent_span_id == root.span_id and span.trace_id == root.trace_id, (
        "the tool span must be a child of the current span: same trace id, parent span id "
        "equal to the parent's span id — start it with tracer.span(...), inside the parent"
    )
    assert span.status_code == STATUS_UNSET, (
        f"a step that succeeded has status code {span.status_code}; leave it unset (0), "
        "do not call set_status(STATUS_OK)"
    )
    assert span.attributes == tool_span_attributes(4, "lookup", result, attempts, backoff), (
        "the span's attributes must be exactly what tool_span_attributes returns for this call, "
        "with the step index that was passed in"
    )

    seen: list[str | None] = []

    def probe(key):
        current = tracer.current_span()
        seen.append(current.name if current else None)
        return {"value": 1}

    probing = ToolRegistry(tools={"lookup": probe}, schemas=TOOL_SCHEMAS)
    with tracer.span("invoke_agent check"):
        traced_call(tracer, probing, 0, "lookup", {"key": "a"}, 0, random.Random(0))
    assert seen == ["execute_tool lookup"], (
        f"the tool saw current span {seen!r} while it ran: the call must happen INSIDE the "
        "span's with-block, or the span times nothing and owns nothing"
    )

    exporter.spans.clear()
    with tracer.span("invoke_agent check"):
        bad, attempts_bad, _ = traced_call(tracer, registry, 3, "look_up", {"key": "a"}, 2,
                                           random.Random(0))
    failed = [s for s in exporter.spans if s.name == "execute_tool look_up"][0]
    assert failed.status_code == STATUS_ERROR and failed.status_message == bad.error, (
        "a step that failed must have status ERROR (2) with the tool's error text as its "
        f"message; got code {failed.status_code}, message {failed.status_message!r}"
    )
    assert failed.attributes.get("error.type") == "unknown_tool", (
        "the error.type from tool_span_attributes must be on the failed span"
    )

    for timeouts in (2, 3):        # recovered on the 3rd attempt, and on the 4th and last
        calls = {"n": 0}

        def flaky(key, calls=calls, timeouts=timeouts):
            calls["n"] += 1
            if calls["n"] <= timeouts:
                raise TransientToolError("lookup timed out")
            return {"value": 9}

        flaky_registry = ToolRegistry(tools={"lookup": flaky}, schemas=TOOL_SCHEMAS)
        exporter.spans.clear()
        with tracer.span("invoke_agent check"):
            result, attempts, _ = traced_call(tracer, flaky_registry, 0, "lookup", {"key": "a"},
                                              3, random.Random(0))
        retried = [s for s in exporter.spans if s.name.startswith("execute_tool")]
        assert result.ok and attempts == timeouts + 1 and len(retried) == 1, (
            f"a step that took {timeouts + 1} attempts must still be ONE span; got "
            f"{len(retried)} spans"
        )
        assert (retried[0].status_code == STATUS_UNSET
                and "error.type" not in retried[0].attributes), (
            f"the {timeouts} timeouts were retried and recovered, so the step's span records no "
            "error, however many attempts it took; the attempts attribute is where they show"
        )
        assert retried[0].attributes["commons.tool.attempts"] == timeouts + 1, (
            "commons.tool.attempts must equal the attempts call_tool_with_retries reported"
        )

    for flag in (True, False):
        want_rng, got_rng = random.Random(3), random.Random(3)
        want = call_tool_with_retries(registry, "lookup", {}, 2, want_rng, validate=flag)
        with tracer.span("invoke_agent check"):
            got = traced_call(tracer, registry, 0, "lookup", {}, 2, got_rng, validate=flag)
        assert got == want, (
            f"with validate={flag} traced_call returned {got!r} but call_tool_with_retries "
            f"returned {want!r}: pass `validate` through to the call"
        )

    plain_rng, traced_rng = random.Random(7), random.Random(7)
    flaky_calls = {"n": 0}

    def flaky_twice(key):
        flaky_calls["n"] += 1
        if flaky_calls["n"] % 3:
            raise TransientToolError("lookup timed out")
        return {"value": 1}

    twice = ToolRegistry(tools={"lookup": flaky_twice}, schemas=TOOL_SCHEMAS)
    expected = call_tool_with_retries(twice, "lookup", {"key": "a"}, 3, plain_rng)
    flaky_calls["n"] = 0
    with tracer.span("invoke_agent check"):
        got = traced_call(tracer, twice, 0, "lookup", {"key": "a"}, 3, traced_rng)
    assert got == expected and plain_rng.getstate() == traced_rng.getstate(), (
        "instrumentation must be invisible to the agent: the same result and the same number "
        "of draws from the `rng` you were given. Do not create a new random.Random here"
    )


_try("exercise 2", _check_traced_call, needs=("exercise 1",))

## 6. The instrumented loop, and one episode as a trace

`run_agent_loop` below is T07-L01's loop with one addition: given a tracer it wraps the
episode in an `invoke_agent {name}` span and sends every tool call through your
`traced_call`. Given no tracer it is byte for byte the original, which is what lets this
lesson prove that watching the agent did not change it.

The root span carries `gen_ai.operation.name` and `gen_ai.agent.name` from the moment it
starts, because the conventions list those as attributes that can drive sampling decisions and
so should be present at span creation. When the episode ends in anything but success, the root
gets status Error and an `error.type` naming the failure, which makes the failure taxonomy a
property of the spans and not something a reader has to reconstruct from logs.

In [ ]:
def _episode(policy: ScriptedPolicy, registry: ToolRegistry, max_steps: int, max_cost: int,
             max_retries: int, stuck_window: int, rng: random.Random, validate: bool,
             tracer: Tracer | None) -> AgentTrace:
    trace: list[dict] = []
    cost_spent = 0
    while True:
        status = budget_status(len(trace), cost_spent, max_steps, max_cost)
        if status is not None:
            return AgentTrace(steps=trace, status=status, cost_spent=cost_spent,
                              n_steps=len(trace))
        tool, args = policy.next_action(trace)
        if tracer is None:
            result, attempts, _backoff = call_tool_with_retries(registry, tool, args,
                                                                max_retries, rng,
                                                                validate=validate)
        else:
            result, attempts, _backoff = traced_call(tracer, registry, len(trace), tool, args,
                                                     max_retries, rng, validate=validate)
        cost_spent += attempts
        trace.append({"tool": tool, "args": args, "ok": result.ok, "error": result.error,
                     "attempts": attempts, "output": result.output})
        if tool == "submit" and result.ok:
            status = "success" if result.output.get("correct") else "wrong_answer"
            return AgentTrace(steps=trace, status=status, cost_spent=cost_spent,
                              n_steps=len(trace))
        if is_stuck(trace, stuck_window):
            return AgentTrace(steps=trace, status="stuck", cost_spent=cost_spent,
                              n_steps=len(trace))


def run_agent_loop(policy: ScriptedPolicy, registry: ToolRegistry, max_steps: int,
                   max_cost: int, max_retries: int, stuck_window: int,
                   rng: random.Random, validate: bool = True,
                   tracer: Tracer | None = None) -> AgentTrace:
    """T07-L01's loop; with a `tracer`, the episode becomes an `invoke_agent` span."""
    if tracer is None:
        return _episode(policy, registry, max_steps, max_cost, max_retries, stuck_window, rng,
                        validate, None)
    with tracer.span(f"invoke_agent {AGENT_NAME}",
                     attributes={"gen_ai.operation.name": "invoke_agent",
                                 "gen_ai.agent.name": AGENT_NAME}) as root:
        trace = _episode(policy, registry, max_steps, max_cost, max_retries, stuck_window, rng,
                         validate, tracer)
        if trace.status != "success":
            root.set_status(STATUS_ERROR)
            root.set_attribute("error.type", trace.status)
    return trace


def run_suite(tasks: list[ChainTask], tracer: Tracer | None = None) -> list[AgentTrace]:
    """Every task in `tasks` through the loop, each with its own seeded random streams."""
    traces = []
    for i, task in enumerate(tasks):
        tool_rng = random.Random(f"{SEED}:{i}:tool")
        policy_rng = random.Random(f"{SEED}:{i}:policy")
        retry_rng = random.Random(f"{SEED}:{i}:retry")
        registry = make_registry(task.chain, task.target, TRANSIENT_RATE, tool_rng)
        plan = [("lookup", {"key": k}) for k in task.key_sequence]
        policy = ScriptedPolicy(plan, ("submit", {"value": task.target}), FAULT_RATE, policy_rng)
        traces.append(run_agent_loop(policy, registry, MAX_STEPS, MAX_COST, MAX_RETRIES,
                                     STUCK_WINDOW, retry_rng, validate=True, tracer=tracer))
    return traces


def _show_one_episode() -> None:
    exporter = ListExporter()
    tracer = Tracer(str(SEED), exporter)
    run_suite(_SUITE, tracer=tracer)
    by_trace: dict[str, list[Span]] = {}
    for sp in exporter.spans:
        by_trace.setdefault(sp.trace_id, []).append(sp)
    # the first episode, in run order, that had a failing step: a rule, not a search
    chosen = next(spans for spans in by_trace.values()
                  if any(s.status_code == STATUS_ERROR for s in spans if s.parent_span_id))
    root = next(s for s in chosen if s.parent_span_id is None)
    print(f"trace {root.trace_id}: {len(chosen)} spans")
    print(f"  {root.name}  status={root.status_code}  error.type={root.attributes.get('error.type', '-')}")
    for sp in sorted((s for s in chosen if s.parent_span_id), key=lambda s: s.start_ns):
        et = sp.attributes.get("error.type")
        print(f"    └ {sp.name:<26} step={sp.attributes['commons.step.index']:<2} "
              f"attempts={sp.attributes['commons.tool.attempts']} status={sp.status_code}"
              + (f" error.type={et}" if et else ""))


_try("one episode as a trace", _show_one_episode, needs=("exercise 1", "exercise 2"))

## 7. Exercise 3 — `encode_attributes()`

A span in memory is a Python object. On the wire, OTLP describes it in protocol buffers, and
the JSON form of the same schema (OTLP/JSON) is what you can read, diff and post to any
receiver that accepts it. Two of its rules catch people out. An attribute is not `{key: value}`
but a list of `{"key": ..., "value": {<one typed field>}}`. And a 64-bit integer is written as a
decimal **string**, because a JSON parser that reads every number as a double cannot hold all
64-bit integers exactly. Booleans are their own type, not integers.

<details><summary>💡 Hint 1 — what to think about</summary>

The value of each attribute is a small object with exactly one field, whose name says the
type: string, boolean, integer, double, or an array of further values. Which of these does
each Python type map to? Python's `bool` is a subclass of `int`: in what order do you ask?
An array holds further values: what does that suggest about how you build it?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Write a small function that encodes one value and call it for each item of the list too.
Test for `bool` before `int`. Turn the integer into its decimal text. An array's field wraps
a list of already-encoded values. For a type you do not support, raise `TypeError` and name
the key, so a bad attribute is noticed at export instead of vanishing.
</details>

In [ ]:
def encode_attributes(attrs: dict[str, Any]) -> list[dict]:
    """A dict of attributes as an OTLP/JSON `attributes` list, in the dict's own order.

    Each item is {"key": name, "value": {...}} where the value has ONE field:
    str -> {"stringValue": s}; bool -> {"boolValue": b}; int -> {"intValue": "<decimal text>"};
    float -> {"doubleValue": x}; list or tuple -> {"arrayValue": {"values": [encoded items]}},
    each item encoded by these same rules. Any other type raises TypeError naming the key.

    Example:
        >>> encode_attributes({"gen_ai.tool.name": "lookup", "commons.tool.attempts": 3})
        [{'key': 'gen_ai.tool.name', 'value': {'stringValue': 'lookup'}}, {'key': 'commons.tool.attempts', 'value': {'intValue': '3'}}]
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_encode_attributes() -> None:
    got = encode_attributes({"s": "lookup", "n": 3, "f": 0.5, "b": True})
    assert got == [{"key": "s", "value": {"stringValue": "lookup"}},
                   {"key": "n", "value": {"intValue": "3"}},
                   {"key": "f", "value": {"doubleValue": 0.5}},
                   {"key": "b", "value": {"boolValue": True}}], (
        f"got {got!r}; each attribute is {{'key': k, 'value': {{one typed field}}}}, in the "
        "dict's order, with the integer written as a decimal STRING"
    )
    flag = encode_attributes({"b": False})[0]["value"]
    assert flag == {"boolValue": False}, (
        f"got {flag!r} for False; a bool is not an integer — test for bool before int"
    )
    big = encode_attributes({"n": 2 ** 62 + 1, "m": -7})
    assert big[0]["value"] == {"intValue": str(2 ** 62 + 1)} and big[1]["value"] == {"intValue": "-7"}, (
        f"got {big!r}; a 64-bit integer must be its exact decimal text, negative sign included"
    )
    arr = encode_attributes({"a": ["x", 2, True]})[0]["value"]
    assert arr == {"arrayValue": {"values": [{"stringValue": "x"}, {"intValue": "2"},
                                             {"boolValue": True}]}}, (
        f"got {arr!r}; an array wraps its items, each encoded by the same rules"
    )
    assert encode_attributes({}) == [], "an empty dict encodes to an empty list"
    try:
        encode_attributes({"bad": {1, 2}})
    except TypeError as exc:
        assert "bad" in str(exc), "the TypeError must name the offending attribute key"
    else:
        raise AssertionError("a set is not an OTLP attribute type: raise TypeError, "
                             "do not encode it silently or drop it")


_try("exercise 3", _check_encode_attributes)

## 8. Exercise 4 — `decode_attributes()`

Whatever receives OTLP/JSON has to read it back. This is the inverse of exercise 3, with one
extra duty: a receiver is told to accept integers as either JSON numbers or strings, because
different exporters emit either. And it must not lose precision: a 64-bit integer that goes
through a float comes back changed.

<details><summary>💡 Hint 1 — what to think about</summary>

Each value has one typed field. Read which field is present and convert accordingly. Which
Python conversion turns the text of an integer into an int without ever passing through a
float? What should an empty value, with no field at all, become?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Look for the field by name and handle each in turn: text stays text, booleans stay booleans,
integers go through the integer constructor whether they arrive as text or as a number, doubles
stay doubles, and an array is a list built by decoding each item the same way. A value with
none of these fields decodes to `None`. Return a dict keyed by attribute key.
</details>

In [ ]:
def decode_attributes(attrs: list[dict]) -> dict[str, Any]:
    """Read an OTLP/JSON `attributes` list back into a dict.

    Inverse of `encode_attributes`: "stringValue" -> str, "boolValue" -> bool, "intValue" ->
    int (whether the JSON holds a string like "3" or a number like 3), "doubleValue" -> float,
    "arrayValue" -> list of decoded items. A value with none of these fields decodes to None.
    An integer must survive exactly, however large.

    Example:
        >>> decode_attributes([{"key": "n", "value": {"intValue": "3"}},
        ...                    {"key": "s", "value": {"stringValue": "lookup"}}])
        {'n': 3, 's': 'lookup'}
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_decode_attributes() -> None:
    wire = [{"key": "s", "value": {"stringValue": "lookup"}},
            {"key": "n", "value": {"intValue": "3"}},
            {"key": "f", "value": {"doubleValue": 0.5}},
            {"key": "b", "value": {"boolValue": False}}]
    got = decode_attributes(wire)
    assert got == {"s": "lookup", "n": 3, "f": 0.5, "b": False}, (
        f"got {got!r}; each typed field decodes to its own Python type"
    )
    assert type(got["n"]) is int and type(got["b"]) is bool, (
        "the integer must come back as an int and the boolean as a bool"
    )
    as_number = decode_attributes([{"key": "n", "value": {"intValue": 7}}])
    assert as_number == {"n": 7}, (
        f"got {as_number!r}; a receiver must accept an integer sent as a JSON number as well "
        "as one sent as a string"
    )
    exact = decode_attributes([{"key": "n", "value": {"intValue": str(2 ** 60 + 1)}}])
    assert exact == {"n": 2 ** 60 + 1}, (
        f"got {exact!r}; converting through float loses the low bits of a large integer — "
        "convert the text with int() directly"
    )
    nested = [{"key": "a", "value": {"arrayValue": {"values": [{"stringValue": "x"},
                                                                 {"intValue": "2"}]}}}]
    assert decode_attributes(nested) == {"a": ["x", 2]}, "an array decodes to a list of decoded items"
    assert decode_attributes([{"key": "e", "value": {}}]) == {"e": None}, (
        "a value with no typed field decodes to None"
    )
    zeros = decode_attributes([{"key": "s", "value": {"stringValue": ""}},
                               {"key": "n", "value": {"intValue": 0}},
                               {"key": "f", "value": {"doubleValue": 0.0}}])
    assert zeros == {"s": "", "n": 0, "f": 0.0}, (
        f"got {zeros!r}: an empty string, a 0 and a 0.0 are values, not missing ones — ask "
        "whether the field is PRESENT, not whether its value is truthy"
    )
    payload = {"s": "x", "n": -12, "big": 2 ** 62, "f": 1.25, "b": True, "l": [1, "two", 3.5]}
    assert decode_attributes(encode_attributes(payload)) == payload, (
        "decode_attributes(encode_attributes(x)) must give x back: check the round trip"
    )


_try("exercise 4", _check_decode_attributes, needs=("exercise 3",))

## 9. One instrumentation, two backends

The point of a standard wire format is that the code above does not know who is listening.
`OtlpJsonExporter` turns a finished trace into one OTLP/JSON payload (a `resourceSpans`
document: resource, then instrumentation scope, then spans) and hands **the same payload** to
every sink it has. Two sinks are given:

- `JsonlSink` keeps each payload as one line of JSON, the shape of a collector's file
  exporter; read it back and you have OTLP spans as plain dicts.
- `SqliteSink` is a different kind of backend: it parses the payload with *your*
  `decode_attributes` and stores one row per span in an SQL table, where the questions are
  asked in SQL.

Real backends are the same idea at a larger scale. Two of them, Langfuse and Phoenix, say they
accept OTLP; this lesson does not contact either, because nothing here may need a network.
What you would send them is the payload printed below.

In [ ]:
def span_to_otlp(sp: Span) -> dict:
    """One `Span` as an OTLP/JSON span object. Integers that are 64-bit go out as strings."""
    out: dict[str, Any] = {
        "traceId": sp.trace_id, "spanId": sp.span_id, "name": sp.name, "kind": sp.kind,
        "startTimeUnixNano": str(sp.start_ns), "endTimeUnixNano": str(sp.end_ns),
        "attributes": encode_attributes(sp.attributes), "status": {"code": sp.status_code},
    }
    if sp.parent_span_id:
        out["parentSpanId"] = sp.parent_span_id     # a root span has none, not an empty one
    if sp.status_message:
        out["status"]["message"] = sp.status_message
    return out


def to_otlp_json(spans: list[Span], service_name: str = SERVICE_NAME) -> dict:
    """A batch of spans as one OTLP/JSON export request."""
    return {"resourceSpans": [{
        "resource": {"attributes": encode_attributes({"service.name": service_name})},
        "scopeSpans": [{"scope": {"name": "commons.t08l01"},
                        "spans": [span_to_otlp(sp) for sp in spans]}],
    }]}


class OtlpJsonExporter:
    """Serialises each finished trace once and hands the same payload to every sink."""

    def __init__(self, sinks: list[Any], service_name: str = SERVICE_NAME):
        self.sinks = sinks
        self.service_name = service_name

    def export(self, spans: list[Span]) -> None:
        payload = to_otlp_json(spans, self.service_name)
        for sink in self.sinks:
            sink.receive(payload)


def _spans_of(payload: dict) -> list[dict]:
    return [span for rs in payload["resourceSpans"] for ss in rs["scopeSpans"]
            for span in ss["spans"]]


class JsonlSink:
    """Backend 1: OTLP/JSON, one payload per line, as a collector's file exporter writes it."""

    def __init__(self) -> None:
        self.lines: list[str] = []

    def receive(self, payload: dict) -> None:
        self.lines.append(json.dumps(payload, separators=(",", ":")))

    def spans(self) -> list[dict]:
        return [span for line in self.lines for span in _spans_of(json.loads(line))]

    def size_bytes(self) -> int:
        return sum(len(line.encode("utf-8")) + 1 for line in self.lines)     # + the newline


class SpanReport(NamedTuple):
    n_episodes: int
    n_steps: int
    mean_steps: float
    tool_errors: int
    recoveries: int
    cost_per_success: float
    failure_taxonomy: dict[str, int]


class SqliteSink:
    """Backend 2: one SQL row per span, in an in-memory database, queried with SQL."""

    def __init__(self) -> None:
        self.db = sqlite3.connect(":memory:")
        self.db.execute("CREATE TABLE spans (trace_id TEXT, span_id TEXT, parent_span_id TEXT, "
                        "name TEXT, operation TEXT, start_ns INTEGER, status_code INTEGER, "
                        "error_type TEXT, step_index INTEGER, attempts INTEGER)")

    def receive(self, payload: dict) -> None:
        for span in _spans_of(payload):
            attrs = decode_attributes(span["attributes"])
            self.db.execute(
                "INSERT INTO spans VALUES (?,?,?,?,?,?,?,?,?,?)",
                (span["traceId"], span["spanId"], span.get("parentSpanId"), span["name"],
                 attrs.get("gen_ai.operation.name"), int(span["startTimeUnixNano"]),
                 span.get("status", {}).get("code", 0), attrs.get("error.type"),
                 attrs.get("commons.step.index"), attrs.get("commons.tool.attempts")))

    def report(self) -> SpanReport:
        one = lambda sql: self.db.execute(sql).fetchone()[0]     # noqa: E731
        episodes = one("SELECT COUNT(*) FROM spans WHERE operation = 'invoke_agent'")
        steps = one("SELECT COUNT(*) FROM spans WHERE operation = 'execute_tool'")
        successes = one("SELECT COUNT(*) FROM spans WHERE operation = 'invoke_agent' "
                        "AND status_code != 2")
        cost = one("SELECT COALESCE(SUM(attempts), 0) FROM spans WHERE operation = 'execute_tool'")
        errors = one("SELECT COUNT(*) FROM spans WHERE operation = 'execute_tool' "
                     "AND status_code = 2")
        recoveries = one(
            "SELECT COUNT(*) FROM spans s WHERE s.operation = 'execute_tool' AND s.status_code = 2 "
            "AND EXISTS (SELECT 1 FROM spans n WHERE n.trace_id = s.trace_id "
            "AND n.operation = 'execute_tool' AND n.start_ns > s.start_ns)")
        taxonomy = dict(self.db.execute(
            "SELECT error_type, COUNT(*) AS c FROM spans WHERE operation = 'invoke_agent' "
            "AND status_code = 2 GROUP BY error_type ORDER BY c DESC, MIN(start_ns)").fetchall())
        return SpanReport(episodes, steps, steps / episodes if episodes else 0.0, errors,
                          recoveries, cost / successes if successes else float("inf"), taxonomy)


def _traced_run() -> dict[str, Any]:
    """The whole suite through the instrumented loop, exported to both sinks at once."""
    kept, jsonl, sql = ListExporter(), JsonlSink(), SqliteSink()
    tracer = Tracer(str(SEED), FanOutExporter([kept, OtlpJsonExporter([jsonl, sql])]))
    traces = run_suite(_SUITE, tracer=tracer)
    return {"traces": traces, "spans": kept.spans, "jsonl": jsonl, "sql": sql}


def _show_the_payload() -> None:
    run = _traced_run()
    docs = [json.loads(line) for line in run["jsonl"].lines]
    first_failed = next(span for doc in docs for span in _spans_of(doc)
                        if span["status"]["code"] == STATUS_ERROR and "parentSpanId" in span)
    print("one payload, as both sinks received it, shortened to one failed step:")
    print(json.dumps(first_failed, indent=1))
    n_spans = sum(len(_spans_of(doc)) for doc in docs)
    rows = run["sql"].db.execute("SELECT COUNT(*) FROM spans").fetchone()[0]
    print(f"sink 1 holds {len(docs)} payloads ({n_spans} spans); "
          f"sink 2 holds {rows} rows: the same spans, seen two ways")


_try("two sinks", _show_the_payload, needs=("exercise 1", "exercise 2", "exercise 3", "exercise 4"))

## 10. Exercise 5 — `report_from_spans()`

Now the reason to instrument: answer the operating questions from the spans alone, with no
access to the agent's own traces. Given the spans a backend holds (as OTLP/JSON dicts), rebuild
the numbers T07-L03 computes from `AgentTrace` objects: episodes, steps, tool errors,
recoveries, cost per success, and the failure taxonomy.

Read spans the way a real backend hands them over. They arrive in **any order**: an exporter
writes spans as they end, a database returns rows as it likes. A field may be **absent** where
it holds its default, because protobuf JSON omits fields that hold their default value. And they are **not all yours**:
an agent shares a trace store with every other service. A *recovery* is a failed step that some
later step of the same trace follows, so "later" has to come from the start times.

<details><summary>💡 Hint 1 — what to think about</summary>

Which attribute tells an episode span from a step span from something else entirely? What
makes an episode a success, given how the loop marked the failures? For recoveries, what do
you need to know about each trace before you can decide whether a step had a successor? Read
the times as integers; they arrive as text.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Decode each span's attributes, sort spans into episodes and steps by the operation attribute,
and ignore the rest. Treat a missing status as unset and a missing parent as a root. Total the
attempts over every step, including those of failed episodes, for the cost; divide by the
number of episodes that did not end in error. Find the latest step start in each trace once,
then count failed steps that start before it. Order the taxonomy by count, then by when each
failure kind first appeared, judged by episode start time.
</details>

In [ ]:
def report_from_spans(spans: list[dict]) -> SpanReport:
    """Recompute the trajectory metrics from OTLP/JSON span dicts alone.

    An *episode* is a span whose "gen_ai.operation.name" is "invoke_agent"; a *step* is one
    whose operation is "execute_tool"; every other span is ignored. Episodes are counted, and
    an episode is a success unless its status code is 2 (a missing "status" or "code" means
    0). `n_steps` counts step spans, and `mean_steps` is `n_steps / n_episodes` (0.0 when there
    are no episodes). `tool_errors` counts steps with status code 2. A *recovery* is such a
    step for which another step of the same "traceId" has a larger integer
    "startTimeUnixNano". `cost_per_success` is the sum of "commons.tool.attempts" over EVERY
    step, divided by the number of successful episodes in one division, as T07-L03 does
    (`float("inf")` if none). The taxonomy counts each failed episode's "error.type" (default
    "_OTHER"), highest count first; kinds with equal counts are ordered by the start time of
    each kind's EARLIEST failed episode. Spans may be in any order.

    Example:
        >>> root = {"traceId": "t1", "spanId": "r", "startTimeUnixNano": "10", "status": {"code": 0},
        ...         "attributes": [{"key": "gen_ai.operation.name", "value": {"stringValue": "invoke_agent"}}]}
        >>> step = {"traceId": "t1", "spanId": "s", "parentSpanId": "r", "startTimeUnixNano": "11",
        ...         "attributes": [{"key": "gen_ai.operation.name", "value": {"stringValue": "execute_tool"}},
        ...                        {"key": "commons.tool.attempts", "value": {"intValue": "2"}}]}
        >>> r = report_from_spans([step, root])
        >>> r.n_episodes, r.n_steps, r.cost_per_success
        (1, 1, 2.0)
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _wire(op: str, trace: str, span: str, start: int, code: int | None = 0, attempts: int = 1,
          error_type: str | None = None, parent: str | None = "root") -> dict:
    """A hand-made OTLP/JSON span, written in the wire format directly."""
    attrs = [{"key": "gen_ai.operation.name", "value": {"stringValue": op}}]
    if op == "execute_tool":
        attrs.append({"key": "commons.tool.attempts", "value": {"intValue": str(attempts)}})
    if error_type:
        attrs.append({"key": "error.type", "value": {"stringValue": error_type}})
    out: dict[str, Any] = {"traceId": trace, "spanId": span, "name": op, "kind": 1,
                           "startTimeUnixNano": str(start), "attributes": attrs}
    if code is not None:
        out["status"] = {"code": code}
    if parent and op == "execute_tool":
        out["parentSpanId"] = parent
    return out


def _check_report_from_spans() -> None:
    spans = [
        _wire("invoke_agent", "A", "a0", 100, 0),
        _wire("execute_tool", "A", "a1", 101, 2, attempts=4, error_type="timeout"),
        _wire("execute_tool", "A", "a2", 102, 0, attempts=1),
        _wire("execute_tool", "A", "a3", 103, 0, attempts=2),
        _wire("invoke_agent", "B", "b0", 200, 2, error_type="wrong_answer"),
        _wire("execute_tool", "B", "b1", 201, 0, attempts=1),
        _wire("execute_tool", "B", "b2", 202, 2, attempts=3, error_type="timeout"),
        _wire("invoke_agent", "C", "c0", 300, 1),
        _wire("execute_tool", "C", "c1", 301, 0, attempts=1),
        _wire("execute_tool", "C", "c2", 302, 0, attempts=1),
    ]
    r = report_from_spans(spans)
    assert (r.n_episodes, r.n_steps) == (3, 7), (
        f"got {r.n_episodes} episodes and {r.n_steps} steps, expected 3 and 7: an episode is "
        "an invoke_agent span, a step is an execute_tool span"
    )
    assert abs(r.mean_steps - 7 / 3) < 1e-12, f"mean_steps is {r.mean_steps}, expected n_steps / n_episodes"
    assert r.tool_errors == 2, f"got {r.tool_errors} tool errors; count steps whose status code is 2"
    assert r.recoveries == 1, (
        f"got {r.recoveries} recoveries, expected 1: step a1 failed and a2 followed it; step "
        "b2 failed but is the last step of ITS trace (trace C ran later, which is irrelevant), "
        "so nothing recovered from it"
    )
    assert r.cost_per_success == 6.5, (
        f"got cost_per_success {r.cost_per_success}, expected 6.5: all 13 attempts over TWO "
        "successful episodes (a root with status code 1 is not a failure; only code 2 is). "
        "Failed episodes cost money too, so their attempts stay in the total"
    )
    assert r.failure_taxonomy == {"wrong_answer": 1}, (
        f"got {r.failure_taxonomy!r}; count the error.type of each episode that ended in error"
    )
    shuffled = spans[::-1]
    assert report_from_spans(shuffled) == r, (
        "the report changed when the same spans arrived in a different order: never rely on the "
        "position of a span in the list, use the start times"
    )
    mixed = spans[3:] + spans[:3]
    assert report_from_spans(mixed) == r, (
        "spans of different traces interleaved or split across the list must give the same "
        "report: group steps by traceId"
    )
    foreign = {"traceId": "Z", "spanId": "z1", "name": "GET /health", "kind": 3,
               "startTimeUnixNano": "5", "attributes": [
                   {"key": "http.request.method", "value": {"stringValue": "GET"}}],
               "status": {"code": 2}}
    assert report_from_spans(spans + [foreign]) == r, (
        "a span that is not part of the agent (another service's span in the same store) must "
        "be ignored, even if it has an error status"
    )
    bare = [_wire("invoke_agent", "C", "c0", 10, None),
            _wire("execute_tool", "C", "c1", 11, None, attempts=3, parent=None),
            _wire("execute_tool", "C", "c2", 12, None, attempts=1, parent=None)]
    rb = report_from_spans(bare)
    assert (rb.n_episodes, rb.tool_errors, rb.cost_per_success) == (1, 0, 4.0), (
        f"got {(rb.n_episodes, rb.tool_errors, rb.cost_per_success)} for spans with no status "
        "and no parentSpanId; an absent status means unset (a success), not a KeyError"
    )
    none_ok = report_from_spans([_wire("invoke_agent", "D", "d0", 1, 2, error_type="stuck"),
                                 _wire("execute_tool", "D", "d1", 2, 0, attempts=2)])
    assert none_ok.cost_per_success == float("inf") and none_ok.failure_taxonomy == {"stuck": 1}, (
        "with no successful episode cost per success is infinite, not zero and not an error"
    )
    last_only = report_from_spans([_wire("invoke_agent", "E", "e0", 1, 2, error_type="stuck"),
                                   _wire("execute_tool", "E", "e1", 2, 0),
                                   _wire("execute_tool", "E", "e2", 3, 2, error_type="timeout")])
    second_last = report_from_spans([_wire("invoke_agent", "F", "f0", 1, 2, error_type="stuck"),
                                     _wire("execute_tool", "F", "f1", 2, 2, error_type="timeout"),
                                     _wire("execute_tool", "F", "f2", 3, 0)])
    assert (last_only.recoveries, second_last.recoveries) == (0, 1), (
        f"a failed final step is 0 recoveries and a failed second-to-last step is 1; got "
        f"{last_only.recoveries} and {second_last.recoveries}: a recovery needs a LATER step"
    )
    digits = report_from_spans([_wire("invoke_agent", "G", "g0", 8, 0),
                                _wire("execute_tool", "G", "g1", 9, 2, error_type="timeout"),
                                _wire("execute_tool", "G", "g2", 10, 0)])
    assert digits.recoveries == 1, (
        "start times arrive as text, and '9' sorts after '10' as text: convert them to int "
        "before comparing"
    )
    t0 = _CLOCK_START_NS                   # a real-sized nanosecond time, like the tracer's
    close = report_from_spans([_wire("invoke_agent", "N", "n0", t0, 0),
                               _wire("execute_tool", "N", "n1", t0 + 1, 2, error_type="timeout"),
                               _wire("execute_tool", "N", "n2", t0 + 2, 0)])
    assert close.recoveries == 1, (
        "two steps 1 ns apart at a real timestamp: through a float they compare equal, because "
        "a double cannot hold every 64-bit integer. Convert the text with int(), as in exercise 4"
    )
    ties = report_from_spans([_wire("invoke_agent", "P", "p0", 50, 2, error_type="step_budget"),
                              _wire("invoke_agent", "M", "m0", 25, 2, error_type="stuck"),
                              _wire("invoke_agent", "Q", "q0", 40, 2),
                              _wire("invoke_agent", "H", "h0", 5, 2, error_type="wrong_answer"),
                              _wire("invoke_agent", "R", "r0", 30, 2, error_type="cost_budget"),
                              _wire("invoke_agent", "L", "l0", 15, 2, error_type="stuck"),
                              _wire("invoke_agent", "W", "w0", 70, 2, error_type="wrong_answer")])
    assert list(ties.failure_taxonomy.items()) == [("wrong_answer", 2), ("stuck", 2),
                                                   ("cost_budget", 1), ("_OTHER", 1),
                                                   ("step_budget", 1)], (
        f"got {list(ties.failure_taxonomy.items())!r}; highest count first, and among equal "
        "counts the failure kind whose EARLIEST episode started first comes first (not its "
        "latest episode, not alphabetical order, not the order the spans arrived in); an "
        "episode with no error.type counts as '_OTHER'"
    )
    assert report_from_spans([]).cost_per_success == float("inf"), (
        "no spans at all must give zero episodes and an infinite cost per success, not a crash"
    )


_try("exercise 5", _check_report_from_spans, needs=("exercise 4",))

## 11. Does it reconcile? Spans against T07-L03's trace metrics

Here is the test of the whole lesson. The suite has run through the instrumented loop and
both sinks have received every trace. Three computations now describe the same run:

1. T07-L03's `trajectory_metrics`, read from the agent's own `AgentTrace` objects;
2. your `report_from_spans`, run on the spans read back from the JSON-lines sink;
3. plain SQL over the second sink's table, which never saw an `AgentTrace` either.

They must agree **exactly**, not approximately: these are counts and ratios of counts, each
ratio computed by one division as T07-L03 computes it, so any difference means a span was
dropped, invented or mislabelled. (Compute a ratio another way, say by multiplying by
`1 / n`, and its last bit can differ with every span in place.) The cell also re-runs the
suite with no tracer at all and compares the two lists of traces, because an instrumented
agent that behaves differently from the uninstrumented one has not been observed, it has been
changed.

In [ ]:
def _show_reconciliation() -> None:
    run = _traced_run()
    traces = run["traces"]
    plain = run_suite(_SUITE)          # the same suite, no tracer anywhere
    print(f"the {len(traces)} traces are identical with and without the tracer: {plain == traces}")
    ref = trajectory_metrics(traces)
    from_file = report_from_spans(run["jsonl"].spans())
    from_sql = run["sql"].report()
    print(f"\n{'metric':<18}{'T07-L03 (traces)':>22}{'spans, JSONL sink':>22}{'spans, SQL sink':>22}  equal")
    every = True
    for name in ("n_episodes", "mean_steps", "tool_errors", "recoveries", "cost_per_success"):
        a, b, c = getattr(ref, name), getattr(from_file, name), getattr(from_sql, name)
        same = a == b == c
        every &= same
        print(f"{name:<18}{a!r:>22}{b!r:>22}{c!r:>22}  {same}")
    tax = failure_taxonomy_from_traces(traces)
    same_tax = (list(tax.items()) == list(from_file.failure_taxonomy.items())
                == list(from_sql.failure_taxonomy.items()))
    every &= same_tax
    print(f"\nfailure taxonomy from the traces: {tax}")
    print(f"failure taxonomy identical from both span sinks: {same_tax}")
    total_steps = sum(tr.n_steps for tr in traces)
    print(f"steps counted from the agent: {total_steps}; from spans: {from_file.n_steps} "
          f"(JSONL) and {from_sql.n_steps} (SQL)")
    print(f"exact agreement on every metric: {every and plain == traces}")


_try("reconciliation", _show_reconciliation,
     needs=("exercise 1", "exercise 2", "exercise 3", "exercise 4", "exercise 5"))

## 12. What does the instrumentation cost?

Every span is data someone stores and pays to move. The cost worth counting is not a timing
on your machine but a **count**: how many spans, how many attributes and how many bytes of
OTLP/JSON one agent step adds. Those depend only on the design of the instrumentation, so they
print the same everywhere, and you can change the design and watch them move.

The conventions also define attributes this lesson leaves off on purpose. The tool's
arguments and result are *opt-in*: they can hold whatever a user typed, and the conventions
warn that they may contain sensitive information. The cell prices what switching the arguments
on would add, so that the privacy decision and the storage decision can be made together.

In [ ]:
def _show_overhead() -> None:
    run = _traced_run()
    spans = run["jsonl"].spans()
    steps = sum(1 for s in spans if "parentSpanId" in s)
    attrs = sum(len(s["attributes"]) for s in spans)
    size = run["jsonl"].size_bytes()
    print(f"{len(spans)} spans, {attrs} attributes and {size} bytes of OTLP/JSON "
          f"for {steps} agent steps")
    print(f"per agent step: {len(spans) / steps:.3f} spans, {attrs / steps:.3f} attributes, "
          f"{size / steps:.1f} bytes")
    extra = 0
    for tr in run["traces"]:
        for step in tr.steps:
            item = encode_attributes({"gen_ai.tool.call.arguments":
                                      json.dumps(step["args"], sort_keys=True)})[0]
            extra += len(json.dumps(item, separators=(",", ":"))) + 1     # + the comma
    print(f"opting in to tool-call arguments would add {extra / steps:.1f} bytes per step "
          f"({100 * extra / size:.1f}% more) before any tool result is captured")


_try("overhead", _show_overhead,
     needs=("exercise 1", "exercise 2", "exercise 3", "exercise 4"))

## 13. What if you only keep the good traces?

Storage is a reason to sample, and "keep the successes, they are the ones that worked" sounds
harmless. Cost per success is a ratio whose numerator includes the spend on runs that failed.
Delete the failed traces and the numerator loses exactly those attempts. The next cell
recomputes the metric from the same spans after dropping every trace whose root span ended in
error.

In [ ]:
def _show_survivorship() -> None:
    run = _traced_run()
    spans = run["jsonl"].spans()
    roots = {s["traceId"]: s for s in spans if "parentSpanId" not in s}
    keep = {tid for tid, root in roots.items() if root["status"]["code"] != STATUS_ERROR}
    kept = [s for s in spans if s["traceId"] in keep]
    full = report_from_spans(spans)
    partial = report_from_spans(kept)
    word = ("lower" if partial.cost_per_success < full.cost_per_success
            else "higher" if partial.cost_per_success > full.cost_per_success else "the same")
    print(f"cost per success from all {full.n_episodes} traces: {full.cost_per_success:.3f} attempts")
    print(f"cost per success from the {partial.n_episodes} successful traces only: "
          f"{partial.cost_per_success:.3f} attempts, {word} than the truth")
    dropped = [s for s in spans if s["traceId"] not in keep and "parentSpanId" in s]
    spent = sum(decode_attributes(s["attributes"])["commons.tool.attempts"] for s in dropped)
    print(f"the dropped traces held {full.n_episodes - partial.n_episodes} episodes and "
          f"{spent} attempts that no longer count against any success")


_try("survivorship", _show_survivorship, needs=("exercise 4", "exercise 5", "exercise 1",
                                                 "exercise 2", "exercise 3"))

## 14. Optional: the same shape in the real SDK

Nothing above needs the OpenTelemetry SDK, and this lesson does not install it. If you have
it (`opentelemetry-sdk`), the next cell makes the same two-level trace with the real API, to
show that `with tracer.start_as_current_span(...)` is the block you wrote by hand. It is
skipped, harmlessly, when the package is absent.

In [ ]:
def _optional_sdk_cell() -> None:
    """Runs the same two-level trace through the real SDK if it is installed; never fails."""
    import logging
    sdk_log = logging.getLogger("opentelemetry")
    level = sdk_log.level
    try:
        import importlib
        import importlib.util
        if importlib.util.find_spec("opentelemetry") is None:
            print("optional: the OpenTelemetry SDK is not installed here, so this cell is skipped")
            return
        sdk_log.setLevel(logging.CRITICAL)      # the SDK logs, with a traceback, what it swallows
        trace_mod = importlib.import_module("opentelemetry.sdk.trace")
        export_mod = importlib.import_module("opentelemetry.sdk.trace.export")

        class _Collect(export_mod.SpanExporter):
            def __init__(self) -> None:
                self.names: list[str] = []

            def export(self, spans):
                self.names.extend(s.name for s in spans)
                return export_mod.SpanExportResult.SUCCESS

        collect = _Collect()
        provider = trace_mod.TracerProvider()
        provider.add_span_processor(export_mod.SimpleSpanProcessor(collect))
        tracer = provider.get_tracer("commons.t08l01")
        with tracer.start_as_current_span(f"invoke_agent {AGENT_NAME}") as root:
            root.set_attribute("gen_ai.operation.name", "invoke_agent")
            with tracer.start_as_current_span("execute_tool lookup") as step:
                step.set_attribute("gen_ai.operation.name", "execute_tool")
        print("the real SDK finished these spans, in this order:", collect.names)
    except Exception as exc:       # an optional cell must never stop the notebook
        print(f"optional SDK cell could not run ({type(exc).__name__}: {exc}); "
              "the lesson does not depend on it")
    finally:
        sdk_log.setLevel(level)


_optional_sdk_cell()

## 15. Common mistakes

- **Writing an integer as a JSON number.** OTLP/JSON writes 64-bit integers as decimal strings,
  and a receiver reading them through a float silently changes large values. Exercises 3 and 4
  check both directions.
- **Setting status OK on success.** Leave the status unset when the operation ended without an
  error. An explicit OK is for a person who has validated the outcome.
- **Recording a recovered retry as an error.** A step that timed out twice and then worked
  did not fail; the retries belong in an attribute, and an error status would turn a healthy
  step into an alert.
- **Putting free text in `error.type`.** It is meant to be low-cardinality. The message
  goes in the status description, which is not a dimension you group by.
- **Trusting the order spans arrive in.** Ask the start times. A backend's row order is not
  the order things happened.
- **Drawing conclusions from a sample that dropped the failures.** Section 13 measures it.
- **Instrumentation that consumes the agent's randomness.** A tracer that draws from the
  same generator changes what the agent does; section 11 compares traced and untraced runs.

In [ ]:
# The first mistake in the list, made visible: a 64-bit integer that takes a trip through a float.
_n = 2 ** 60 + 1
_back = int(float(_n))
print(f"{_n} through a float comes back as {_back}: "
      f"{'the same integer' if _back == _n else 'a different integer'}")
print("decimal text loses nothing:", int(str(_n)) == _n)

## 16. Self-check

Answers come with this lesson's worked solution when you enrol on Synapsa.

1. In OTLP/JSON the attribute `commons.tool.attempts` with the value 3 is written
   `{"intValue": "3"}`, with the 3 in quotes. Why?
   - (a) JSON has no integer type at all, so every number in an OTLP payload has to travel
     inside quotes
   - (b) 64-bit integers can exceed what a JSON reader that parses every number as a double
     holds exactly, so the protobuf JSON mapping writes them as decimal strings
   - (c) OTLP is a text protocol, so every value in it must be quoted, whatever its type
   - (d) The quotes mark the attribute as a counter rather than a gauge, which is how a
     backend knows to sum it

2. A team stores only the traces that ended in success and recomputes cost per success from
   them. Compared with the truth, the figure they report is:
   - (a) the same, because failed runs produce no useful spans and so add nothing to either
     side of the ratio
   - (b) higher, because successful runs are the expensive ones and the failed ones stop early
   - (c) lower, because the attempts spent on failed episodes have left the total
   - (d) undefined, because a ratio like this needs at least one failed run to divide by

3. A tool call times out twice and succeeds on the third attempt. Following the recording-errors
   guidance, the step's span should have:
   - (a) its status left unset, with the retries visible through an attempts attribute
   - (b) status Error and `error.type` set to the timeout, so the two timeouts show in the
     error rate
   - (c) three separate spans, one per attempt made, the first two with status Error
   - (d) status OK, set explicitly, to show that the retry worked in the end

4. `report_from_spans` sorts steps by start time instead of using their position in the list.
   Why does that matter for counting recoveries?
   - (a) Python lists cannot be trusted to keep their order once passed to a function
   - (b) The trace ids are random, so two spans of different traces cannot be told apart
   - (c) It makes the function run faster on large lists, because sorted data is cheaper to scan
   - (d) A recovery is defined by a later step, and the order spans arrive in says nothing
     about which step came later

In [ ]:
_MARKS = {"passed": "✅", "failed": "❌", "not started": "⏳"}


def _progress_board() -> None:
    width = max(len(", ".join(funcs)) for funcs in _EXERCISES.values())
    print("progress board")
    for label, funcs in _EXERCISES.items():
        state = _STATUS.get(label, "not started")
        print(f"  {_MARKS[state]} {label:<12} {', '.join(funcs):<{width}}  {state}")
    done = sum(_STATUS.get(label) == "passed" for label in _EXERCISES)
    print(f"\n{done} of {len(_EXERCISES)} exercises complete")
    failing = [label for label in _EXERCISES if _STATUS.get(label) == "failed"]
    if failing:
        print("failing right now: " + ", ".join(failing) + ". Each one printed what went "
              "wrong in its own cell above, and every exercise has hints you can open.")
    elif done < len(_EXERCISES):
        print("work top to bottom: every exercise has hints you can open above its code.")

## What you built

`tool_span_attributes` and `traced_call` turned the agent loop into an OpenTelemetry trace
that follows the GenAI conventions and leaves the agent's behaviour untouched.
`encode_attributes` and `decode_attributes` are the two halves of OTLP/JSON, and the same
payload reached two backends that share nothing but that format. `report_from_spans` answered
the operating questions from the spans alone, and section 11 showed it agreeing exactly with
T07-L03's metrics. Sections 12 and 13 put numbers on what the instrumentation costs and on how
a sampling shortcut misleads. The next step is the same instrumentation pointed at a model that
spends tokens instead of attempts.

In [ ]:
# Your progress board. Every check is re-run here, quietly, against your code as it stands
# now — each one already printed its feedback in its own cell above — so the board is
# current even if you edited an exercise and did not re-run its check.
if __name__ == "__main__":
    _ALL_CHECKS = (
        ("exercise 1", _check_tool_span_attributes, ()),
        ("exercise 2", _check_traced_call, ("exercise 1",)),
        ("exercise 3", _check_encode_attributes, ()),
        ("exercise 4", _check_decode_attributes, ("exercise 3",)),
        ("exercise 5", _check_report_from_spans, ("exercise 4",)),
    )
    with contextlib.redirect_stdout(io.StringIO()):
        for _name, _check, _needs in _ALL_CHECKS:
            _try(_name, _check, needs=_needs)
    _progress_board()
    if _FAILED_CHECKS and "ipykernel" not in sys.modules:
        raise SystemExit("checks failed: " + ", ".join(dict.fromkeys(_FAILED_CHECKS)))

<!-- COMMONS NOTICE v1 · generated by tools/notebooks.py · do not edit by hand -->
---
**Synapsa Commons** · © 2026 RealAI · licensed under [CC BY-NC-SA
4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)

**You may** use this lesson to learn and to teach, and copy, fork, share and adapt it.

**You must** credit "Synapsa Commons by RealAI" with a link to
https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials,
say what you changed, and share anything you adapt under this same licence.

**You may not** use it, or anything adapted from it, in a way primarily intended for
commercial advantage or payment: for example selling it, charging for a course, bootcamp or
training built on it, or packaging it into a paid product or service. For a commercial
licence, contact [RealAI](https://www.realai.eu/contact).

Third-party material in this lesson keeps its own licence, named in `assets/SOURCE.md` or
`claims.yaml`. The Synapsa name and logo belong to RealAI and are not licensed. This summary
is not the licence: the [legal
code](https://creativecommons.org/licenses/by-nc-sa/4.0/legalcode) governs.